# Held-Out Evaluation: Base Agent vs Skills-Enhanced Agent

This notebook runs a controlled comparison of both agents on a held-out evaluation dataset,
using the optimized prompt from `06-PromptOptimization.ipynb` as the shared system prompt.

Both agents use the **identical optimized prompt** (loaded from `@production`), so the only
variable is the presence of agent skills. This isolates the incremental contribution of skills
from the gains achieved by prompt optimization alone.

The evaluation uses `mlflow.genai.evaluate()` with the aligned judge from `05-JudgeAlignment.ipynb`,
and results are logged directly to the MLflow experiment.

**Prerequisites:**
- `03_create_agent_definition.ipynb` has written `agent.py`
- `04-Evaluation.ipynb` has created the evaluation dataset
- `05-JudgeAlignment.ipynb` has produced the aligned judge
- `06-PromptOptimization.ipynb` has optimized and registered the prompt to `@production`
- `07-AgentSkillsGeneration.ipynb` has generated skills
- `08_create_agent_with_skills.ipynb` has written `agent_with_skills.py`

In [ ]:
%pip install -U -qqqq backoff databricks-openai uv databricks-agents "mlflow>=3.9" dspy databricks-mcp langgraph-checkpoint-postgres "psycopg[binary,pool]" databricks-langchain langgraph
dbutils.library.restartPython()

In [ ]:
import json
import os
import hashlib
import importlib
import warnings
import logging
from contextlib import contextmanager
from pathlib import Path

import mlflow

CONFIG = json.loads(Path("config/atbat_assistant.json").read_text())

EXPERIMENT_ID = CONFIG["mlflow"]["experiment_id"]
PROMPT_NAME = CONFIG["prompt_registry"]["prompt_name"]
ALIGNED_JUDGE_NAME = CONFIG["judges"]["aligned_judge_name"]

JUDGE_EXPERIMENT_ID = EXPERIMENT_ID

_scope = CONFIG["prompt_registry_auth"]["secret_scope_name"]
_sp_id = dbutils.secrets.get(scope=_scope, key=CONFIG["prompt_registry_auth"]["oauth_client_id_key"])
_sp_secret = dbutils.secrets.get(scope=_scope, key=CONFIG["prompt_registry_auth"]["oauth_client_secret_key"])

@contextmanager
def _sp_auth():
    """Temporarily swap to Service Principal OAuth for prompt registry operations."""
    saved_token = os.environ.pop("DATABRICKS_TOKEN", None)
    os.environ["DATABRICKS_CLIENT_ID"] = _sp_id
    os.environ["DATABRICKS_CLIENT_SECRET"] = _sp_secret
    try:
        yield
    finally:
        os.environ.pop("DATABRICKS_CLIENT_ID", None)
        os.environ.pop("DATABRICKS_CLIENT_SECRET", None)
        if saved_token is not None:
            os.environ["DATABRICKS_TOKEN"] = saved_token

with _sp_auth():
    parent_experiment = mlflow.get_experiment(EXPERIMENT_ID)
    experiment_09 = mlflow.set_experiment(f"{parent_experiment.name}-09-held-out-evaluation")
EXPERIMENT_ID = experiment_09.experiment_id
print(f"Using experiment: {experiment_09.name} (ID: {EXPERIMENT_ID})")
print(f"Aligned judge loaded from: {JUDGE_EXPERIMENT_ID}")

## Step 1: Load Optimized Prompt

Load the best prompt from `@production` (registered by `06-PromptOptimization.ipynb`).

In [ ]:
with _sp_auth():
    prompt_obj = mlflow.genai.load_prompt(f"prompts:/{PROMPT_NAME}@production")

optimized_prompt_text = prompt_obj.format()
print(f"Loaded prompt: {PROMPT_NAME} v{prompt_obj.version} (@production)")
print(f"Prompt hash: {hashlib.md5(optimized_prompt_text.encode()).hexdigest()[:12]}")
print(f"Prompt length: {len(optimized_prompt_text)} chars")
print(f"\nFirst 300 chars:\n{optimized_prompt_text[:300]}...")

## Step 2: Load Aligned Judge

In [ ]:
from mlflow.genai.scorers import get_scorer, Scorer
from mlflow.entities import Feedback
import re as _re
import sys as _sys
import litellm as _litellm

GUARDRAIL_SENTINEL = -1.0
_GUARDRAIL_RE = _re.compile(r'(?<![_a-zA-Z])arsenal(?![_a-zA-Z])', _re.IGNORECASE)


def _scrub(text):
    """Replace guardrail-triggering baseball terms in a string."""
    if isinstance(text, str):
        return _GUARDRAIL_RE.sub('pitch repertoire', text)
    return text


def _scrub_messages(messages):
    if not messages or not isinstance(messages, list):
        return
    for msg in messages:
        if not isinstance(msg, dict):
            continue
        c = msg.get("content")
        if isinstance(c, str):
            msg["content"] = _scrub(c)
        elif isinstance(c, list):
            for part in c:
                if isinstance(part, dict) and isinstance(part.get("text"), str):
                    part["text"] = _scrub(part["text"])


_original_completion = _litellm.completion


def _sanitized_completion(*args, **kwargs):
    _scrub_messages(kwargs.get("messages"))
    if args:
        for a in args:
            if isinstance(a, list):
                _scrub_messages(a)
    return _original_completion(*args, **kwargs)


_litellm.completion = _sanitized_completion

_patched_count = 0
for _mod_name in list(_sys.modules.keys()):
    _mod = _sys.modules.get(_mod_name)
    if _mod is None:
        continue
    for _attr in ['completion', 'litellm_completion']:
        try:
            _ref = getattr(_mod, _attr, None)
            if _ref is _original_completion:
                setattr(_mod, _attr, _sanitized_completion)
                _patched_count += 1
        except Exception:
            pass

print(f"Patched litellm.completion + {_patched_count} cached refs across loaded modules")

_inner_judge = get_scorer(name=ALIGNED_JUDGE_NAME, experiment_id=JUDGE_EXPERIMENT_ID)
print(f"Loaded aligned judge: {_inner_judge.name}")

_patched_count2 = 0
for _mod_name in list(_sys.modules.keys()):
    _mod = _sys.modules.get(_mod_name)
    if _mod is None:
        continue
    for _attr in ['completion', 'litellm_completion']:
        try:
            _ref = getattr(_mod, _attr, None)
            if _ref is _original_completion:
                setattr(_mod, _attr, _sanitized_completion)
                _patched_count2 += 1
        except Exception:
            pass
if _patched_count2:
    print(f"Patched {_patched_count2} additional refs found after scorer load")


class GuardrailSafeScorer(Scorer):
    """Wraps an existing scorer and catches AI Gateway guardrail errors.

    Returns Feedback with value=GUARDRAIL_SENTINEL (-1.0) so downstream
    consumers can detect and exclude these from aggregation.
    Also scrubs inputs/outputs before delegating and detects predict_fn fallbacks.
    """
    name: str = ALIGNED_JUDGE_NAME
    _delegate: object = None

    class Config:
        underscore_attrs_are_private = True

    def __init__(self, delegate, **kwargs):
        super().__init__(**kwargs)
        self._delegate = delegate

    def __call__(self, *, inputs=None, outputs=None, expectations=None, trace=None):
        if isinstance(outputs, str) and "(Skipped: input guardrail triggered" in outputs:
            logging.warning("predict_fn returned guardrail fallback, marking as sentinel")
            return Feedback(
                name=self.name,
                value=GUARDRAIL_SENTINEL,
                rationale="GUARDRAIL_SKIP",
            )
        scrubbed_outputs = _scrub(outputs) if isinstance(outputs, str) else outputs
        try:
            return self._delegate(
                inputs=inputs, outputs=scrubbed_outputs,
                expectations=expectations, trace=trace,
            )
        except Exception as e:
            if "guardrail" in str(e).lower() or "input_guardrail_triggered" in str(e):
                logging.warning(f"Guardrail triggered during scoring: {str(e)[:120]}")
                return Feedback(
                    name=self.name,
                    value=GUARDRAIL_SENTINEL,
                    rationale="GUARDRAIL_SKIP",
                )
            raise


aligned_judge = GuardrailSafeScorer(delegate=_inner_judge)
print(f"Wrapped judge with GuardrailSafeScorer (sentinel={GUARDRAIL_SENTINEL})")

## Step 3: Define Evaluation Helpers

In [ ]:
warnings.filterwarnings('ignore')
warnings.filterwarnings('ignore', category=UserWarning, module='pydantic')
logging.getLogger('mlflow.genai.judges.instructions_judge').setLevel(logging.ERROR)
logging.getLogger('mlflow.tracing.fluent').setLevel(logging.ERROR)
logging.getLogger('mlflow.tracing.export.mlflow_v3').setLevel(logging.ERROR)
logging.getLogger('mlflow.tracing.provider').setLevel(logging.ERROR)


def _get(item, key, default=""):
    if isinstance(item, dict):
        return item.get(key, default)
    return getattr(item, key, default)


def _extract_compact_response(result) -> str:
    tool_calls = []
    final_text = ""

    for item in result.output:
        item_type = _get(item, "type")

        if item_type == "function_call":
            name = _get(item, "name", "unknown")
            args_str = _get(item, "arguments", "{}")
            if len(args_str) > 300:
                args_str = args_str[:300] + "..."
            tool_calls.append(f"  - {name}({args_str})")

        elif item_type == "message":
            content = _get(item, "content", [])
            if isinstance(content, list):
                for block in content:
                    block_type = _get(block, "type") if isinstance(block, dict) else getattr(block, "type", "")
                    if block_type == "output_text":
                        text = _get(block, "text", "") if isinstance(block, dict) else getattr(block, "text", "")
                        if text:
                            final_text = text
            elif isinstance(content, str):
                final_text = content

        elif item_type == "text":
            text = _get(item, "text", "")
            if text:
                final_text = text

    parts = []
    if tool_calls:
        parts.append("[Tool Calls]\n" + "\n".join(tool_calls))
    if final_text:
        parts.append("[Agent Analysis]\n" + final_text)

    compact = "\n\n".join(parts) if parts else "(no response)"
    return _scrub(compact)


def eval_predict_fn_factory(agent_module_name, prompt_text):
    """Create a predict function for mlflow.genai.evaluate().

    Tracing stays enabled so evaluate() captures full traces.
    """
    with _sp_auth():
        mod = importlib.import_module(agent_module_name)
        agent_instance = mod.AGENT

    def predict_fn(input):
        if isinstance(input, dict) and "input" in input:
            user_message = input["input"][0]["content"]
        elif isinstance(input, list):
            user_message = input[0]["content"]
        else:
            user_message = str(input)

        messages = [
            {"role": "system", "content": prompt_text},
            {"role": "user", "content": user_message},
        ]
        try:
            result = agent_instance.predict({"input": messages})
            return _extract_compact_response(result)
        except Exception as e:
            err_str = str(e)
            if "input_guardrail_triggered" in err_str or "guardrail" in err_str.lower():
                logging.warning(f"Guardrail triggered during eval: {user_message[:80]}")
                return "(Skipped: input guardrail triggered)"
            raise

    return predict_fn


print("Evaluation helpers defined.")

## Step 4: Load Evaluation Dataset

Load the held-out evaluation dataset created by `04-Evaluation.ipynb`.

In [ ]:
from mlflow.genai.datasets import get_dataset

EVAL_DATASET_NAME = CONFIG["evaluation"]["dataset_name"]
eval_ds = get_dataset(name=EVAL_DATASET_NAME)
eval_df = eval_ds.to_df()

print(f"Dataset columns: {list(eval_df.columns)}")
print(f"First row keys: {eval_df.iloc[0].to_dict().keys()}")

eval_data = []
for _, row in eval_df.iterrows():
    inputs = row.get("inputs")
    if inputs is None:
        continue
    if isinstance(inputs, str):
        inputs = json.loads(inputs)
    if isinstance(inputs, dict) and "input" in inputs:
        entry = {"inputs": inputs}
    else:
        entry = {"inputs": {"input": inputs}}

    expectations = row.get("expectations")
    if expectations is not None:
        if isinstance(expectations, str):
            expectations = json.loads(expectations)
        if expectations:
            entry["expectations"] = expectations

    eval_data.append(entry)

print(f"Loaded {len(eval_data)} evaluation records from dataset '{EVAL_DATASET_NAME}'")
print(f"Sample inputs keys: {list(eval_data[0]['inputs'].keys())}")
has_expectations = any("expectations" in e for e in eval_data)
print(f"Has expectations: {has_expectations}")
if has_expectations:
    print(f"Sample expectations: {eval_data[0].get('expectations', {})}")

## Step 5: Diagnostic -- test scorer on one example

Run evaluate on a single row to verify the scorer produces metrics before launching the full run.

In [ ]:
from mlflow.genai import evaluate

diag_pfn = eval_predict_fn_factory("agent", optimized_prompt_text)

print("Running single-row diagnostic...")
diag_result = evaluate(
    data=eval_data[:1],
    predict_fn=diag_pfn,
    scorers=[aligned_judge],
)

print("\n" + "=" * 60)
print("FULL RETURN OBJECT INSPECTION")
print("=" * 60)
print(f"Type: {type(diag_result)}")
print(f"Dir:  {[a for a in dir(diag_result) if not a.startswith('_')]}")
for attr in [a for a in dir(diag_result) if not a.startswith('_')]:
    try:
        val = getattr(diag_result, attr)
        if not callable(val):
            val_str = str(val)
            if len(val_str) > 500:
                val_str = val_str[:500] + "..."
            print(f"\n  .{attr} = {val_str}")
    except Exception as e:
        print(f"\n  .{attr} -> ERROR: {e}")

print("\n" + "=" * 60)
print("TRACE INSPECTION")
print("=" * 60)
diag_traces = mlflow.search_traces(run_id=diag_result.run_id)
print(f"Traces found: {len(diag_traces)}")
if len(diag_traces) > 0:
    row = diag_traces.iloc[0]
    print(f"  Columns: {list(diag_traces.columns)}")
    print(f"  Status: {row.get('status', 'N/A')}")
    print(f"  Response preview: {str(row.get('response', ''))[:300]}")
    assessments = row.get("assessments", [])
    print(f"  Assessments: {len(assessments) if assessments else 0}")
    if assessments:
        for a in assessments:
            print(f"    - {a}")

if not diag_result.metrics:
    print("\nWARNING: Metrics are empty. Check details above.")
    print("Continuing to full evaluation anyway.")
else:
    print("\nScorer is working. Proceeding to full evaluation.")

## Step 6: Run Full Evaluation

Evaluate both agents on the complete held-out dataset.

In [ ]:
os.environ["MLFLOW_GENAI_EVAL_MAX_WORKERS"] = "3"
os.environ["MLFLOW_GENAI_EVAL_MAX_SCORER_WORKERS"] = "1"
print("Concurrency: max_workers=3, max_scorer_workers=1")

eval_results = {}
for agent_module in ["agent", "agent_with_skills"]:
    label = "base" if agent_module == "agent" else "skills"
    print(f"\n{'=' * 60}")
    print(f"Evaluating: {agent_module} (label: {label})")
    print(f"{'=' * 60}")

    pfn = eval_predict_fn_factory(agent_module, optimized_prompt_text)

    result = evaluate(
        data=eval_data,
        predict_fn=pfn,
        scorers=[aligned_judge],
    )

    eval_results[label] = result
    print(f"  Metrics: {result.metrics}")

    if hasattr(result, "tables") and result.tables:
        for tname, tdf in result.tables.items():
            print(f"  Table '{tname}': {len(tdf)} rows, columns: {list(tdf.columns)}")

print(f"\n{'=' * 60}")
print("HELD-OUT EVALUATION COMPLETE")
print(f"{'=' * 60}")

## Step 7: Summarize Results

In [ ]:
import numpy as np

CHECKPOINT_TABLE = f"{CONFIG['workspace']['catalog']}.{CONFIG['workspace']['schema']}.gepa_experiment_checkpoint"


def _extract_scores_from_traces(run_id, scorer_name):
    """Pull per-row scores from trace assessments when result.metrics is empty."""
    traces_df = mlflow.search_traces(run_id=run_id)
    scores = []
    for _, row in traces_df.iterrows():
        for a in (row.get("assessments") or []):
            if a.get("assessment_name") == scorer_name:
                val = a.get("feedback", {}).get("value")
                if val == "yes":
                    scores.append(1.0)
                elif val == "no":
                    scores.append(0.0)
                elif isinstance(val, (int, float)):
                    scores.append(float(val))
    return scores


print("GEPA OPTIMIZATION RESULTS (from 06-PromptOptimization)")
print("=" * 90)
if spark.catalog.tableExists(CHECKPOINT_TABLE):
    import pandas as pd
    _cp_df = spark.table(CHECKPOINT_TABLE).filter("agent_type = 'base'").toPandas()
    _cp_df["lift"] = _cp_df["final_score"] / _cp_df["initial_score"]
    print(_cp_df[["run_idx", "initial_score", "final_score", "lift"]].to_string(index=False, float_format="%.4f"))
    print(f"\n  Mean initial (1-5): {_cp_df['initial_score'].mean()*5:.2f} +/- {_cp_df['initial_score'].std()*5:.2f}")
    print(f"  Mean final (1-5):   {_cp_df['final_score'].mean()*5:.2f} +/- {_cp_df['final_score'].std()*5:.2f}")
else:
    print(f"  (Checkpoint table {CHECKPOINT_TABLE} not found)")

print(f"\n{'=' * 90}")
print("HELD-OUT EVALUATION (same optimized prompt, both agents)")
print("=" * 90)
for label in ["base", "skills"]:
    if label not in eval_results:
        continue
    result = eval_results[label]
    metrics = result.metrics
    if metrics:
        print(f"  {label:10s}: {metrics}")
    else:
        scores = _extract_scores_from_traces(result.run_id, ALIGNED_JUDGE_NAME)
        valid = [s for s in scores if s != GUARDRAIL_SENTINEL]
        if valid:
            print(f"  {label:10s}: mean={np.mean(valid):.4f}, n={len(valid)}, guardrail_skips={len(scores)-len(valid)}")
        else:
            print(f"  {label:10s}: no valid scores found (raw scores: {scores[:5]})")

print(f"\n{'=' * 90}")
print("SUMMARY FOR PAPER")
print("=" * 90)
if spark.catalog.tableExists(CHECKPOINT_TABLE):
    print(f"\n1) Pre-optimization baseline (n={len(_cp_df)}):")
    print(f"   Score (1-5): {_cp_df['initial_score'].mean()*5:.2f} +/- {_cp_df['initial_score'].std()*5:.2f}")
    print(f"\n2) Align + GEPA (n={len(_cp_df)}):")
    print(f"   Score (1-5): {_cp_df['final_score'].mean()*5:.2f} +/- {_cp_df['final_score'].std()*5:.2f}")
    gain = ((_cp_df['final_score'].mean() - _cp_df['initial_score'].mean()) / _cp_df['initial_score'].mean()) * 100
    print(f"   Gain: {gain:.1f}%")

print(f"\n3) Held-out evaluation (prompt v{prompt_obj.version}, {EVAL_DATASET_NAME} dataset):")
for label in ["base", "skills"]:
    if label not in eval_results:
        continue
    result = eval_results[label]
    if result.metrics:
        print(f"   {label}: {result.metrics}")
    else:
        scores = _extract_scores_from_traces(result.run_id, ALIGNED_JUDGE_NAME)
        valid = [s for s in scores if s != GUARDRAIL_SENTINEL]
        if valid:
            print(f"   {label}: mean={np.mean(valid):.4f} +/- {np.std(valid):.4f} (n={len(valid)}, skips={len(scores)-len(valid)})")
        else:
            print(f"   {label}: no valid scores")

print(f"\nFull traces and per-example scores are in MLflow experiment: {experiment_09.name}")

## Held-Out Score Comparison Chart

Visualize aligned-judge scores across the three evaluation configurations:
1. **Baseline** - Original agent with optimized prompt
2. **Optimized Prompt** - GEPA-optimized prompt only
3. **Optimized Prompt + Skills** - GEPA-optimized prompt with generated agent skills

In [ ]:
import json
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import mlflow

CONFIG = json.loads(Path("config/atbat_assistant.json").read_text())
ALIGNED_JUDGE_NAME = CONFIG["judges"]["aligned_judge_name"]
GUARDRAIL_SENTINEL = -1.0

RUN_IDS = {
    "Baseline (Original Agent)": "<your_run_id>",
    "Optimized Prompt": "<your_run_id>",
    "Optimized Prompt + Skills": "<your_run_id>",
}

SCORER_NAME = ALIGNED_JUDGE_NAME


def _extract_scores_from_run(run_id, scorer_name):
    """Pull per-row numeric scores from trace assessments for a given run."""
    experiment_id = mlflow.get_run(run_id).info.experiment_id
    traces_df = mlflow.search_traces(run_id=run_id, locations=[experiment_id])
    scores = []
    for _, row in traces_df.iterrows():
        for a in (row.get("assessments") or []):
            if a.get("assessment_name") == scorer_name:
                val = a.get("feedback", {}).get("value")
                if val == "yes":
                    scores.append(1.0)
                elif val == "no":
                    scores.append(0.0)
                elif isinstance(val, (int, float)):
                    scores.append(float(val))
    return scores


scores_by_config = {}
for label, run_id in RUN_IDS.items():
    raw = _extract_scores_from_run(run_id, SCORER_NAME)
    valid = [s for s in raw if s != GUARDRAIL_SENTINEL]
    scores_by_config[label] = valid
    print(f"{label}: mean={np.mean(valid):.2f}, std={np.std(valid):.2f}, n={len(valid)}, skipped={len(raw)-len(valid)}")

labels = list(scores_by_config.keys())
means = [np.mean(v) for v in scores_by_config.values()]
stds = [np.std(v) for v in scores_by_config.values()]
counts = [len(v) for v in scores_by_config.values()]

colors = ["#6B7280", "#3B82F6", "#10B981"]

fig, ax = plt.subplots(figsize=(8, 5))
bars = ax.bar(labels, means, yerr=stds, capsize=6, color=colors, edgecolor="white", linewidth=1.2, width=0.55)

for bar, mean, std, n in zip(bars, means, stds, counts):
    ax.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + std + 0.06,
        f"{mean:.2f}\n(n={n})",
        ha="center", va="bottom", fontsize=11, fontweight="bold",
    )

ax.set_ylabel("Aligned Judge Score (1-5)", fontsize=12)
ax.set_title("Held-Out Evaluation: Aligned Judge Scores by Configuration", fontsize=13, fontweight="bold")
ax.set_ylim(0, 5.0)
ax.yaxis.set_major_locator(ticker.MultipleLocator(1.0))
ax.yaxis.set_minor_locator(ticker.MultipleLocator(0.5))
ax.grid(axis="y", alpha=0.3, linestyle="--")
ax.set_axisbelow(True)

baseline_mean = means[0]
for i, (bar, mean) in enumerate(zip(bars, means)):
    if i > 0 and baseline_mean > 0:
        pct_gain = ((mean - baseline_mean) / baseline_mean) * 100
        ax.text(
            bar.get_x() + bar.get_width() / 2,
            0.15,
            f"+{pct_gain:.1f}%",
            ha="center", va="bottom", fontsize=10, color="white", fontweight="bold",
        )

plt.tight_layout()
plt.show()